<a href="https://colab.research.google.com/github/zsgwu/G5_GWU_CAPST/blob/main/get_embeddings_eduyou.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EduYou — Get Embeddings (Azure OpenAI)

**File:** `get_embeddings_eduyou.ipynb`  
**Generated:** 2026-03-30

This notebook is a drop-in adaptation of the professor’s *get_embeddings* template for the EduYou capstone. It creates vector embeddings for **one row per document** from:

- `cleaned/eduyou_cip_docs_for_embedding.csv`

and writes an embeddings table compatible with the professor’s `07_RAG_query.ipynb` pattern (doc_id + `dim_*` columns).

---

## Column-by-column mapping (EduYou → RAG template)

| EduYou CSV column | Role in RAG template | Used how |
|---|---|---|
| `text` | document text | embedded into a vector |
| `doc_id` | document identifier | stored alongside embedding |
| `cip4`, `degree_level`, `cip_title`, `median_earnings_4yr_nat` | metadata | carried into output for filtering/debugging |

> ✅ **Important:** Do *not* embed the raw multi-million-row join tables. Use the document-level file (`eduyou_cip_docs_for_embedding.csv`).

---

## Embedding model configuration (Professor note)

Set `AZURE_OPENAI_DEPLOYMENT_ID` to one of:
- `text-embedding-ada-002`
- `text-embedding-3-small`
- `text-embedding-3-large`

Deployments are created using the same names as the models, so you can use those names directly.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print('Working dir:', os.getcwd())
print('Files:', os.listdir())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working dir: /content
Files: ['.config', 'embeddings', 'drive', 'sample_data']


## 1) Install dependencies


In [ ]:
!pip -q install openai pandas numpy

## 2) Configuration

Set your Azure OpenAI endpoint and key securely.
- Recommended: store in environment variables.
- Do **not** hardcode secrets in notebooks.


In [ ]:
import os
import getpass
import pandas as pd
import numpy as np
from openai import AzureOpenAI

# --- Required settings ---
AZURE_OPENAI_ENDPOINT = (
    os.getenv('AZURE_OPENAI_ENDPOINT')
    or getpass.getpass('AZURE_OPENAI_ENDPOINT (e.g., https://<resource>.openai.azure.com/): ')
)
AZURE_OPENAI_API_KEY = (
    os.getenv('AZURE_OPENAI_API_KEY')
    or getpass.getpass('AZURE_OPENAI_API_KEY: ')
)

# ✅ Updated to match professor instructions
AZURE_OPENAI_API_VERSION = (
    os.getenv('AZURE_OPENAI_API_VERSION')
    or '2025-04-01-preview'
)

# --- Deployment/model name (per professor note) ---
AZURE_OPENAI_DEPLOYMENT_ID = (
    os.getenv('AZURE_OPENAI_DEPLOYMENT_ID')
    or 'text-embedding-3-small'
)

# Optional: for text-embedding-3-* you may set dimensions to shorten vectors.
# Leave as None to use model default (3-small: 1536, 3-large: 3072).
EMBEDDING_DIMENSIONS = None

# Input/output paths
OUTPUT_DIR = '/content/drive/MyDrive/group-5/RAG_data/embeddings'
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_EMB_CSV = os.path.join(
    OUTPUT_DIR,
    f'eduyou_embeddings_{AZURE_OPENAI_DEPLOYMENT_ID}.csv'
)

print('Model/deployment:', AZURE_OPENAI_DEPLOYMENT_ID)
print('Input docs:', INPUT_DOCS_CSV)
print('Output embeddings:', OUTPUT_EMB_CSV)



AZURE_OPENAI_ENDPOINT (e.g., https://<resource>.openai.azure.com/): ··········
AZURE_OPENAI_API_KEY: ··········
Model/deployment: text-embedding-3-small
Input docs: /content/drive/MyDrive/group-5/RAG_data/cleaned/eduyou_cip_docs_for_embedding.csv
Output embeddings: /content/drive/MyDrive/group-5/RAG_data/embeddings/eduyou_embeddings_text-embedding-3-small.csv


## 3) Load EduYou document table

This file must have at minimum:
- `doc_id`
- `text`


In [ ]:
docs = pd.read_csv(INPUT_DOCS_CSV, low_memory=False)

required_cols = {'doc_id', 'text'}
missing = required_cols - set(docs.columns)
if missing:
    raise ValueError(f"Missing required columns in {INPUT_DOCS_CSV}: {missing}")

print('Rows (documents):', len(docs))
print('Columns:', docs.columns.tolist())

Rows (documents): 1312
Columns: ['doc_id', 'cip4', 'degree_level', 'cip_title', 'median_earnings_4yr_nat', 'text']


## 4) Create embeddings (batched)

This follows the professor’s Azure OpenAI embedding example using `AzureOpenAI`.
We batch requests to reduce overhead.

The output file uses:
- `doc_id`
- metadata columns (if present)
- embedding columns named `dim_0 ... dim_{p-1}`

The notebook can resume if partially completed.


In [ ]:
import os

print("Embeddings file exists?", os.path.exists(OUTPUT_EMB_CSV))
if os.path.exists(OUTPUT_EMB_CSV):
    os.remove(OUTPUT_EMB_CSV)
    print("✅ Deleted old embeddings file:", OUTPUT_EMB_CSV)

Embeddings file exists? True
✅ Deleted old embeddings file: /content/drive/MyDrive/group-5/RAG_data/embeddings/eduyou_embeddings_text-embedding-3-small.csv


In [ ]:
# client initialization
client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT
)

# Metadata columns to carry through if present
meta_cols = [
    c for c in ['cip4', 'degree_level', 'cip_title', 'median_earnings_4yr_nat']
    if c in docs.columns
]

BATCH_SIZE = 64

# Resume support
if os.path.exists(OUTPUT_EMB_CSV):
    emb_df = pd.read_csv(OUTPUT_EMB_CSV)
    done_ids = set(emb_df['doc_id'].astype(str))
    print(f"Found existing embeddings file with {len(done_ids)} rows. Skipping completed docs.")
else:
    done_ids = set()

# Helper to call embeddings API
def get_embeddings_batch(text_list):
    kwargs = {
        'model': AZURE_OPENAI_DEPLOYMENT_ID,
        'input': text_list,
    }
    if EMBEDDING_DIMENSIONS is not None:
        kwargs['dimensions'] = int(EMBEDDING_DIMENSIONS)

    resp = client.embeddings.create(**kwargs)
    return [d.embedding for d in resp.data]

# Determine embedding dimension once
first_vec = get_embeddings_batch([docs.loc[0, 'text']])[0]
P = len(first_vec)
print('Embedding dimension P =', P)

dim_cols = [f'dim_{i}' for i in range(P)]

# Initialize output file if needed
if not os.path.exists(OUTPUT_EMB_CSV):
    cols = ['doc_id', 'text'] + meta_cols + dim_cols
    pd.DataFrame(columns=cols).to_csv(OUTPUT_EMB_CSV, index=False)

# ✅ FIXED batch loop (no overlapping batches)
for batch_start in range(0, len(docs), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(docs))
    batch_rows = docs.iloc[batch_start:batch_end].copy()

    batch_rows['doc_id'] = batch_rows['doc_id'].astype(str)
    batch_rows = batch_rows[~batch_rows['doc_id'].isin(done_ids)]

    if batch_rows.empty:
        continue

    texts = batch_rows['text'].astype(str).tolist()
    vecs = get_embeddings_batch(texts)

    out_rows = []
    for (_, row), vec in zip(batch_rows.iterrows(), vecs):
        rec = {
    'doc_id': row['doc_id'],
    'text': row['text']
      }
        for mc in meta_cols:
            rec[mc] = row.get(mc, '')
        rec.update({dim_cols[j]: float(vec[j]) for j in range(P)})
        out_rows.append(rec)
        done_ids.add(row['doc_id'])

    pd.DataFrame(out_rows).to_csv(
        OUTPUT_EMB_CSV, mode='a', header=False, index=False
    )

print('✅ Done. Saved embeddings to:', OUTPUT_EMB_CSV)

Embedding dimension P = 1536
✅ Done. Saved embeddings to: /content/drive/MyDrive/group-5/RAG_data/embeddings/eduyou_embeddings_text-embedding-3-small.csv


## 5) Quick sanity check

Load the embeddings output and confirm shape.


In [ ]:
emb = pd.read_csv(OUTPUT_EMB_CSV)
print('Embedding table shape:', emb.shape)
emb.head()

Embedding table shape: (1312, 1542)


,doc_id,text,cip4,degree_level,cip_title,median_earnings_4yr_nat,dim_0,dim_1,dim_2,dim_3,...,dim_1526,dim_1527,dim_1528,dim_1529,dim_1530,dim_1531,dim_1532,dim_1533,dim_1534,dim_1535
0,1001_Associates_Degree,CIP family (cip4): 1001\nField of study: Commu...,1001,Associate's Degree,Communications Technologies/Technicians.,35292.0,-0.004667,-0.000457,0.025152,0.003684,...,0.014464,0.041138,-0.019319,-0.013813,0.038378,0.008501,-0.024488,-0.044471,-0.016651,0.039029
1,1001_Bachelors_Degree,CIP family (cip4): 1001\nField of study: Commu...,1001,Bachelor's Degree,Communications Technologies/Technicians.,36451.0,-0.005267,0.000425,0.030816,0.008580,...,0.015434,0.041010,-0.025667,-0.008638,0.040383,0.007214,-0.021341,-0.050263,-0.015016,0.039990
2,1001_Masters_Degree,CIP family (cip4): 1001\nField of study: Commu...,1001,Master's Degree,Communications Technologies/Technicians.,71506.0,-0.009140,-0.001422,0.022625,-0.010154,...,0.020621,0.039186,-0.018591,-0.012444,0.039577,0.007331,-0.024589,-0.054122,-0.018162,0.030912
3,1001_Undergraduate_Certificate_or_Diploma,CIP family (cip4): 1001\nField of study: Commu...,1001,Undergraduate Certificate or Diploma,Communications Technologies/Technicians.,33445.0,-0.003143,0.000553,0.033031,0.006359,...,0.016846,0.042480,-0.026944,-0.008245,0.035307,0.007967,-0.020287,-0.046371,-0.015007,0.041289
4,1002_Associates_Degree,CIP family (cip4): 1002\nField of study: Audio...,1002,Associate's Degree,Audiovisual Communications Technologies/Techni...,35660.0,-0.025641,-0.005078,0.018319,-0.000279,...,0.018797,0.036689,-0.023196,-0.008914,0.050299,0.000183,-0.025770,-0.046288,-0.017504,0.036767


## Next step

Use the resulting embeddings CSV in the professor’s `07_RAG_query.ipynb` workflow:
- Load this embeddings file
- Load the same document file (`eduyou_cip_docs_for_embedding.csv`)
- Compute query embedding with the same `AZURE_OPENAI_DEPLOYMENT_ID`
- Retrieve Top‑K by distance/similarity
